# 风险归因完整教程：风险的因子来源分解

## 📚 教学目标
1. 理解**边际风险贡献 (MCTR)** 的定义：$\text{MCTR}_k = \frac{(B\Sigma_F)_k}{\sigma_p}$
2. 掌握**风险分解公式**：$\sigma_p^2 = B\Sigma_F B' + D$
3. 在微型数据集上**手算** 2 个因子的风险贡献
4. 使用 **numpy 矩阵运算**验证
5. 可视化风险贡献的**饼图**和**时序图**

**参考书**：《因子投资：方法与实践》（石川）第 5.3 节
**教学策略**：先用极小数据集让你"看见"每一步计算，再扩展到真实规模

---

## 1. 场景设定：风险从哪里来？

### 🎯 一个直觉问题

假设你管理一个投资组合，本月波动率为 **15%**。

老板问你："这 15% 的风险是从哪里来的？是大盘波动带来的？还是小盘股的特异性风险？"

这就是**风险归因**要回答的问题：

> **将组合风险分解为各个因子的贡献，弄清楚风险的来源。**

### 📖 书中的定义

> 风险归因的核心思想是：将投资组合的方差分解为因子风险和特异性风险两部分。
> 通过边际风险贡献（MCTR），可以量化每个因子对组合风险的贡献。

### 🔗 本节的逻辑链条

```
多因子模型: σ_p² = B Σ_F B' + D
    ↓
因子风险 = B Σ_F B'
    ↓
特异性风险 = D
    ↓
边际风险贡献: MCTR_k = (B Σ_F)_k / σ_p
    ↓
风险贡献_k = β_p,k × MCTR_k
    ↓
验证: Σ(风险贡献_k) + 特异性风险 = σ_p²
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# 设置中文字体和样式
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")

---

## 2. 风险归因的数学框架

### 📐 多因子模型的方差分解

投资组合的方差可以分解为：

$$\sigma_p^2 = B \Sigma_F B' + D$$

其中：
- $\sigma_p^2$ = 组合方差
- $B$ = $K \times 1$ 的因子暴露向量
- $\Sigma_F$ = $K \times K$ 的因子协方差矩阵
- $D$ = 特异性风险（对角矩阵的对角线元素之和）

### 📐 因子风险的分解

因子风险部分 $B \Sigma_F B'$ 可以进一步分解为各因子的贡献：

$$B \Sigma_F B' = \sum_{k=1}^{K} \beta_{p,k} \times (\Sigma_F B)_k$$

其中 $(\Sigma_F B)_k$ 是向量 $\Sigma_F B$ 的第 $k$ 个元素。

### 📐 边际风险贡献 (MCTR)

**边际风险贡献 (Marginal Contribution to Risk, MCTR)** 定义为：

$$\text{MCTR}_k = \frac{(B \Sigma_F)_k}{\sigma_p} = \frac{\partial \sigma_p}{\partial \beta_{p,k}}$$

含义：**因子暴露 $\beta_{p,k}$ 每增加 1 个单位，组合标准差增加 $\text{MCTR}_k$ 个单位**。

### 📐 因子风险贡献

每个因子对组合方差的贡献为：

$$\text{RC}_k = \beta_{p,k} \times (\Sigma_F B)_k = \beta_{p,k} \times \text{MCTR}_k \times \sigma_p$$

### 📐 风险归因恒等式

$$\sigma_p^2 = \sum_{k=1}^{K} \text{RC}_k + D$$

因此：

$$\frac{\sum_{k} \text{RC}_k}{\sigma_p^2} + \frac{D}{\sigma_p^2} = 1$$

### 💡 关键特征

- MCTR 是一个**边际量**，衡量因子暴露变化对风险的影响
- 风险贡献是一个**绝对量**，衡量因子对组合方差的贡献
- 特异性风险 $D$ 是不能被因子解释的风险

---

## 3. 微型数据集：10 只股票的风险归因

### 🎯 场景

假设我们管理一个由 **10 只股票**组成的投资组合（等权配置）。
我们使用 **双因子模型**进行风险归因：

| 因子 | 含义 |
|------|------|
| MKT | 市场因子 — 大盘波动的影响 |
| SMB | 市值因子 — 小盘股 vs 大盘股 |

### 📐 数据设定

- 因子协方差矩阵 $\Sigma_F$
- 每只股票的因子暴露 $\beta_{\text{MKT},i}$ 和 $\beta_{\text{SMB},i}$
- 每只股票的特异性风险 $\sigma_{\varepsilon,i}^2$

In [ ]:
# ========== 构造微型数据集 ==========
np.random.seed(42)

# 10 只股票
stocks = [f'S{i+1:02d}' for i in range(10)]
N_STOCKS = 10
N_FACTORS = 2

# 因子暴露矩阵 (N_STOCKS × N_FACTORS)
beta_MKT = np.array([0.8, 1.0, 1.2, 0.9, 1.1, 1.3, 0.7, 1.0, 1.1, 0.9])
beta_SMB = np.array([1.2, 0.5, -0.3, 0.8, -0.5, 1.0, 0.3, -0.2, 0.6, -0.8])
B = np.column_stack([beta_MKT, beta_SMB])  # (10, 2)

# 因子协方差矩阵 Σ_F
sigma_MKT = 4.0  # 市场因子标准差 (%)
sigma_SMB = 2.5  # 市值因子标准差 (%)
rho = 0.3        # 因子相关系数

Sigma_F = np.array([
    [sigma_MKT**2, rho * sigma_MKT * sigma_SMB],
    [rho * sigma_MKT * sigma_SMB, sigma_SMB**2]
])

# 特异性风险（每只股票的特异性方差）
sigma_eps = np.array([3.0, 2.5, 4.0, 3.5, 2.0, 4.5, 2.8, 3.2, 3.8, 2.2])
D_diag = sigma_eps**2  # 特异性方差

# 等权组合的因子暴露
w = np.ones(N_STOCKS) / N_STOCKS  # 等权权重
beta_p = B.T @ w  # (2,) 组合因子暴露

print("📋 微型数据集：")
print("─" * 65)
df_micro = pd.DataFrame({
    'Stock': stocks,
    'Beta_MKT': beta_MKT,
    'Beta_SMB': beta_SMB,
    'Sigma_Eps': sigma_eps
})
print(df_micro.to_string(index=False))

print(f"\n📐 因子协方差矩阵 Σ_F：")
print(f"  σ_MKT = {sigma_MKT}%, σ_SMB = {sigma_SMB}%, ρ = {rho}")
print(f"  Σ_F = [{Sigma_F[0,0]:.2f}  {Sigma_F[0,1]:.2f}]")
print(f"       [{Sigma_F[1,0]:.2f}  {Sigma_F[1,1]:.2f}]")

print(f"\n📐 组合因子暴露（等权平均）：")
print(f"  β_p,MKT = {beta_p[0]:.4f}")
print(f"  β_p,SMB = {beta_p[1]:.4f}")

---

## 4. 手算风险归因：逐步分解

### 📐 计算步骤

**步骤 1**：计算组合方差的因子风险部分

$$\sigma_{p,\text{factor}}^2 = B \Sigma_F B'$$

**步骤 2**：计算组合方差的特异性风险部分

$$\sigma_{p,\text{specific}}^2 = w' D w = \sum_{i} w_i^2 \sigma_{\varepsilon,i}^2$$

**步骤 3**：计算组合总方差

$$\sigma_p^2 = \sigma_{p,\text{factor}}^2 + \sigma_{p,\text{specific}}^2$$

**步骤 4**：计算 MCTR

$$\text{MCTR}_k = \frac{(\Sigma_F \beta_p)_k}{\sigma_p}$$

**步骤 5**：计算各因子的风险贡献

$$\text{RC}_k = \beta_{p,k} \times (\Sigma_F \beta_p)_k$$

In [ ]:
# ========== 手算风险归因 ==========

print("📊 步骤 1: 计算因子风险部分")
print("─" * 55)
# σ²_p,factor = β_p' Σ_F β_p
Sigma_F_beta = Sigma_F @ beta_p  # (2,)
var_factor = beta_p @ Sigma_F @ beta_p
print(f"  Σ_F × β_p = [{Sigma_F_beta[0]:.4f}, {Sigma_F_beta[1]:.4f}]")
print(f"  σ²_p,factor = β_p' × Σ_F × β_p = {var_factor:.4f}")

print(f"\n📊 步骤 2: 计算特异性风险部分")
print("─" * 55)
# σ²_p,specific = Σ w_i² × σ²_ε,i
var_specific = np.sum(w**2 * D_diag)
print(f"  w = [{w[0]:.4f}, {w[1]:.4f}, ..., {w[-1]:.4f}] (等权)")
print(f"  σ²_p,specific = Σ w_i² × σ²_ε,i = {var_specific:.4f}")

print(f"\n📊 步骤 3: 计算组合总方差")
print("─" * 55)
var_total = var_factor + var_specific
sigma_p = np.sqrt(var_total)
print(f"  σ²_p = σ²_p,factor + σ²_p,specific")
print(f"       = {var_factor:.4f} + {var_specific:.4f}")
print(f"       = {var_total:.4f}")
print(f"  σ_p = √{var_total:.4f} = {sigma_p:.4f}%")

print(f"\n📊 步骤 4: 计算 MCTR")
print("─" * 55)
MCTR = Sigma_F_beta / sigma_p
print(f"  MCTR_MKT = (Σ_F × β_p)_MKT / σ_p = {Sigma_F_beta[0]:.4f} / {sigma_p:.4f} = {MCTR[0]:.4f}")
print(f"  MCTR_SMB = (Σ_F × β_p)_SMB / σ_p = {Sigma_F_beta[1]:.4f} / {sigma_p:.4f} = {MCTR[1]:.4f}")

print(f"\n📊 步骤 5: 计算各因子的风险贡献")
print("─" * 55)
RC_MKT = beta_p[0] * Sigma_F_beta[0]
RC_SMB = beta_p[1] * Sigma_F_beta[1]
print(f"  RC_MKT = β_p,MKT × (Σ_F×β_p)_MKT = {beta_p[0]:.4f} × {Sigma_F_beta[0]:.4f} = {RC_MKT:.4f}")
print(f"  RC_SMB = β_p,SMB × (Σ_F×β_p)_SMB = {beta_p[1]:.4f} × {Sigma_F_beta[1]:.4f} = {RC_SMB:.4f}")
print(f"  因子风险合计 = {RC_MKT + RC_SMB:.4f}")

print(f"\n📊 步骤 6: 验证风险归因恒等式")
print("─" * 55)
print(f"  σ²_p = RC_MKT + RC_SMB + D")
print(f"  {var_total:.4f} = {RC_MKT:.4f} + {RC_SMB:.4f} + {var_specific:.4f}")
print(f"  {var_total:.4f} = {RC_MKT + RC_SMB + var_specific:.4f}")
print(f"\n  ✅ 验证通过！")

print(f"\n💡 风险归因解读：")
print(f"  📐 市场因子 (MKT) 贡献了 {RC_MKT:.4f} 的方差 ({RC_MKT/var_total*100:.1f}%)")
print(f"  📐 市值因子 (SMB) 贡献了 {RC_SMB:.4f} 的方差 ({RC_SMB/var_total*100:.1f}%)")
print(f"  📐 特异性风险 贡献了 {var_specific:.4f} 的方差 ({var_specific/var_total*100:.1f}%)")
print(f"  📐 因子风险占比 = {(RC_MKT+RC_SMB)/var_total*100:.1f}%")
print(f"  📐 特异性风险占比 = {var_specific/var_total*100:.1f}%")

---

## 5. 使用 numpy 矩阵运算验证

### 📐 矩阵运算的优势

使用 numpy 的矩阵运算可以：
1. 更简洁地表达数学公式
2. 更高效地处理大规模数据
3. 减少手算错误

In [ ]:
# ========== numpy 矩阵运算验证 ==========
print("📊 numpy 矩阵运算验证")
print("═" * 55)

# 方法 1: 直接计算
# 组合方差 = w' (B Σ_F B' + diag(σ²_ε)) w
B_Sigma_B = B @ Sigma_F @ B.T  # (10, 10) 因子协方差矩阵
D_matrix = np.diag(D_diag)       # (10, 10) 特异性方差矩阵
V_total = B_Sigma_B + D_matrix   # (10, 10) 总协方差矩阵

var_total_matrix = w @ V_total @ w
sigma_p_matrix = np.sqrt(var_total_matrix)

print(f"  方法 1: 直接计算")
print(f"    σ²_p = w' × (B Σ_F B' + D) × w = {var_total_matrix:.4f}")
print(f"    σ_p = {sigma_p_matrix:.4f}%")

# 方法 2: 分解计算
var_factor_matrix = w @ B_Sigma_B @ w
var_specific_matrix = w @ D_matrix @ w

print(f"\n  方法 2: 分解计算")
print(f"    σ²_p,factor = w' × B Σ_F B' × w = {var_factor_matrix:.4f}")
print(f"    σ²_p,specific = w' × D × w = {var_specific_matrix:.4f}")
print(f"    σ²_p = {var_factor_matrix:.4f} + {var_specific_matrix:.4f} = {var_factor_matrix + var_specific_matrix:.4f}")

# 方法 3: 使用组合因子暴露
var_factor_beta = beta_p @ Sigma_F @ beta_p

print(f"\n  方法 3: 使用组合因子暴露")
print(f"    σ²_p,factor = β_p' × Σ_F × β_p = {var_factor_beta:.4f}")

# 验证三种方法的结果一致
print(f"\n🔬 验证三种方法的结果：")
print(f"  方法 1: {var_total_matrix:.6f}")
print(f"  方法 2: {var_factor_matrix + var_specific_matrix:.6f}")
print(f"  手算:   {var_total:.6f}")
print(f"  ✅ 三种方法结果一致！")

---

## 6. 可视化：风险贡献的饼图

### 🎯 直观展示各因子的风险贡献

In [ ]:
# ========== 可视化：风险贡献饼图 ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 左图：方差贡献饼图 ---
ax1 = axes[0]
var_components = [RC_MKT, RC_SMB, var_specific]
labels_var = [f'MKT\n({RC_MKT/var_total*100:.1f}%)', 
              f'SMB\n({RC_SMB/var_total*100:.1f}%)', 
              f'Specific\n({var_specific/var_total*100:.1f}%)']
colors_var = ['#3498db', '#2ecc71', '#95a5a6']

wedges1, texts1, autotexts1 = ax1.pie(var_components, labels=labels_var, colors=colors_var,
                                       autopct='%1.1f%%', startangle=90, pctdistance=0.75,
                                       textprops={'fontsize': 10})
ax1.set_title('Variance Contribution Breakdown', fontsize=13, fontweight='bold')

# --- 右图：标准差贡献（MCTR 视角）---
ax2 = axes[1]
# MCTR 贡献 = β_p,k × MCTR_k
mctr_contrib_MKT = beta_p[0] * MCTR[0]
mctr_contrib_SMB = beta_p[1] * MCTR[1]
# 特异性风险对标准差的贡献
specific_contrib_sigma = var_specific / sigma_p

sigma_components = [mctr_contrib_MKT, mctr_contrib_SMB, specific_contrib_sigma]
labels_sigma = [f'MKT\n({mctr_contrib_MKT/sigma_p*100:.1f}%)', 
                f'SMB\n({mctr_contrib_SMB/sigma_p*100:.1f}%)', 
                f'Specific\n({specific_contrib_sigma/sigma_p*100:.1f}%)']

wedges2, texts2, autotexts2 = ax2.pie(sigma_components, labels=labels_sigma, colors=colors_var,
                                       autopct='%1.1f%%', startangle=90, pctdistance=0.75,
                                       textprops={'fontsize': 10})
ax2.set_title('Standard Deviation Contribution', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  左图：方差贡献的构成，MKT 是最大的风险来源")
print(f"  右图：标准差贡献的构成（MCTR 视角），比例略有不同")
print(f"  特异性风险占比 = {var_specific/var_total*100:.1f}%，说明分散化降低了特异性风险")

---

## 7. 扩展到 50 只股票 × 60 个月

### 🎯 目标

现在我们扩展到**真实规模**，每个月：
1. 生成 50 只股票的截面数据
2. 计算等权组合的风险归因
3. 收集 60 个月的时间序列
4. 分析风险归因的稳定性

### 📐 DGP

- 50 只股票，等权配置
- 2 个因子：MKT, SMB
- 因子暴露固定，因子收益率每月变化
- 特异性风险每月变化

In [ ]:
# ========== 50 只股票 × 60 个月的模拟 ==========
np.random.seed(42)
N_STOCKS_SIM = 50
N_MONTHS_SIM = 60

# 每只股票的因子暴露（固定）
beta_MKT_sim = np.random.normal(1.0, 0.3, N_STOCKS_SIM)
beta_SMB_sim = np.random.normal(0.0, 0.5, N_STOCKS_SIM)
B_sim = np.column_stack([beta_MKT_sim, beta_SMB_sim])  # (50, 2)

# 等权权重
w_sim = np.ones(N_STOCKS_SIM) / N_STOCKS_SIM

# 组合因子暴露（固定）
beta_p_sim = B_sim.T @ w_sim  # (2,)

# 存储结果
var_total_series = []
var_factor_series = []
var_specific_series = []
RC_MKT_series = []
RC_SMB_series = []
MCTR_MKT_series = []
MCTR_SMB_series = []

print(f"📊 模拟参数：")
print(f"  股票数量: {N_STOCKS_SIM} 只/月")
print(f"  时间跨度: {N_MONTHS_SIM} 个月")
print(f"  因子: MKT, SMB")
print(f"  组合因子暴露: β_p,MKT = {beta_p_sim[0]:.4f}, β_p,SMB = {beta_p_sim[1]:.4f}")
print(f"\n🔄 开始逐月计算...")

for t in range(N_MONTHS_SIM):
    # 每月的因子波动率（略有变化）
    sigma_MKT_t = np.random.normal(4.0, 0.5)
    sigma_SMB_t = np.random.normal(2.5, 0.3)
    rho_t = np.random.normal(0.3, 0.1)
    
    # 因子协方差矩阵
    Sigma_F_t = np.array([
        [sigma_MKT_t**2, rho_t * sigma_MKT_t * sigma_SMB_t],
        [rho_t * sigma_MKT_t * sigma_SMB_t, sigma_SMB_t**2]
    ])
    
    # 每月的特异性风险（略有变化）
    sigma_eps_t = np.random.normal(3.0, 0.5, N_STOCKS_SIM)
    sigma_eps_t = np.maximum(sigma_eps_t, 1.0)  # 确保为正
    
    # 因子风险
    Sigma_F_beta_t = Sigma_F_t @ beta_p_sim
    var_factor_t = beta_p_sim @ Sigma_F_t @ beta_p_sim
    
    # 特异性风险
    var_specific_t = np.sum(w_sim**2 * sigma_eps_t**2)
    
    # 总方差
    var_total_t = var_factor_t + var_specific_t
    sigma_p_t = np.sqrt(var_total_t)
    
    # MCTR
    MCTR_t = Sigma_F_beta_t / sigma_p_t
    
    # 因子风险贡献
    RC_MKT_t = beta_p_sim[0] * Sigma_F_beta_t[0]
    RC_SMB_t = beta_p_sim[1] * Sigma_F_beta_t[1]
    
    # 存储
    var_total_series.append(var_total_t)
    var_factor_series.append(var_factor_t)
    var_specific_series.append(var_specific_t)
    RC_MKT_series.append(RC_MKT_t)
    RC_SMB_series.append(RC_SMB_t)
    MCTR_MKT_series.append(MCTR_t[0])
    MCTR_SMB_series.append(MCTR_t[1])

var_total_arr = np.array(var_total_series)
var_factor_arr = np.array(var_factor_series)
var_specific_arr = np.array(var_specific_series)
RC_MKT_arr = np.array(RC_MKT_series)
RC_SMB_arr = np.array(RC_SMB_series)
MCTR_MKT_arr = np.array(MCTR_MKT_series)
MCTR_SMB_arr = np.array(MCTR_SMB_series)
sigma_p_arr = np.sqrt(var_total_arr)

print(f"\n✅ 完成！共收集 {N_MONTHS_SIM} 个月的数据")

In [ ]:
# ========== 统计汇总 ==========
print("📊 风险归因的统计汇总（60 个月）")
print("═" * 70)
print(f"  {'指标':>12} {'σ_p':>10} {'σ²_p':>10} {'σ²_factor':>10} {'σ²_specific':>12}")
print(f"  {'─'*12} {'─'*10} {'─'*10} {'─'*10} {'─'*12}")
print(f"  {'均值':>12} {sigma_p_arr.mean():>10.4f} {var_total_arr.mean():>10.4f} {var_factor_arr.mean():>10.4f} {var_specific_arr.mean():>12.4f}")
print(f"  {'标准差':>12} {sigma_p_arr.std():>10.4f} {var_total_arr.std():>10.4f} {var_factor_arr.std():>10.4f} {var_specific_arr.std():>12.4f}")

print(f"\n📊 因子风险贡献统计：")
print("─" * 55)
print(f"  {'指标':>10} {'RC_MKT':>12} {'RC_SMB':>12} {'因子占比':>10}")
print(f"  {'─'*10} {'─'*12} {'─'*12} {'─'*10}")
factor_ratio = var_factor_arr / var_total_arr
print(f"  {'均值':>10} {RC_MKT_arr.mean():>12.4f} {RC_SMB_arr.mean():>12.4f} {factor_ratio.mean()*100:>10.1f}%")
print(f"  {'标准差':>10} {RC_MKT_arr.std():>12.4f} {RC_SMB_arr.std():>12.4f} {factor_ratio.std()*100:>10.1f}%")

print(f"\n📊 MCTR 统计：")
print("─" * 55)
print(f"  {'指标':>10} {'MCTR_MKT':>12} {'MCTR_SMB':>12}")
print(f"  {'─'*10} {'─'*12} {'─'*12}")
print(f"  {'均值':>10} {MCTR_MKT_arr.mean():>12.4f} {MCTR_SMB_arr.mean():>12.4f}")
print(f"  {'标准差':>10} {MCTR_MKT_arr.std():>12.4f} {MCTR_SMB_arr.std():>12.4f}")

print(f"\n💡 关键发现：")
print(f"  1. 因子风险占比均值 = {factor_ratio.mean()*100:.1f}%，是组合风险的主要来源")
print(f"  2. MKT 因子的风险贡献大于 SMB（因为 β_p,MKT 更大且 σ_MKT 更大）")
print(f"  3. 特异性风险占比 = {(1-factor_ratio.mean())*100:.1f}%，分散化有效降低了特异性风险")

---

## 8. 可视化：风险归因的时序图

### 🎯 展示风险归因随时间的变化

In [ ]:
# ========== 可视化：风险归因时序图 ==========
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

months = np.arange(1, N_MONTHS_SIM + 1)

# --- 图1: 组合波动率时序 ---
ax1 = axes[0, 0]
ax1.plot(months, sigma_p_arr, 'b-', linewidth=1.5, alpha=0.8)
ax1.axhline(y=sigma_p_arr.mean(), color='r', linestyle='--', alpha=0.5, 
            label=f'Mean = {sigma_p_arr.mean():.2f}%')
ax1.set_xlabel('Month', fontsize=11)
ax1.set_ylabel('Portfolio Std Dev (%)', fontsize=11)
ax1.set_title('Portfolio Volatility Over Time', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)

# --- 图2: 风险贡献堆叠面积图 ---
ax2 = axes[0, 1]
ax2.fill_between(months, 0, RC_MKT_arr, alpha=0.7, label='MKT', color='#3498db')
ax2.fill_between(months, RC_MKT_arr, RC_MKT_arr + RC_SMB_arr, 
                 alpha=0.7, label='SMB', color='#2ecc71')
ax2.fill_between(months, RC_MKT_arr + RC_SMB_arr, var_total_arr,
                 alpha=0.7, label='Specific', color='#95a5a6')
ax2.plot(months, var_total_arr, 'k-', linewidth=1.5, label='Total Var', alpha=0.8)
ax2.set_xlabel('Month', fontsize=11)
ax2.set_ylabel('Variance', fontsize=11)
ax2.set_title('Risk Contribution Over Time (Stacked)', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10, loc='upper left')
ax2.grid(alpha=0.3)

# --- 图3: 因子风险占比时序 ---
ax3 = axes[1, 0]
ax3.plot(months, factor_ratio * 100, 'b-', linewidth=1.5, alpha=0.8)
ax3.axhline(y=factor_ratio.mean()*100, color='r', linestyle='--', alpha=0.5,
            label=f'Mean = {factor_ratio.mean()*100:.1f}%')
ax3.set_xlabel('Month', fontsize=11)
ax3.set_ylabel('Factor Risk Ratio (%)', fontsize=11)
ax3.set_title('Factor Risk Ratio Over Time', fontsize=13, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(alpha=0.3)

# --- 图4: MCTR 时序 ---
ax4 = axes[1, 1]
ax4.plot(months, MCTR_MKT_arr, 'b-', linewidth=1.5, alpha=0.8, label='MCTR_MKT')
ax4.plot(months, MCTR_SMB_arr, 'g-', linewidth=1.5, alpha=0.8, label='MCTR_SMB')
ax4.set_xlabel('Month', fontsize=11)
ax4.set_ylabel('MCTR', fontsize=11)
ax4.set_title('Marginal Contribution to Risk', fontsize=13, fontweight='bold')
ax4.legend(fontsize=10)
ax4.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  图1：组合波动率随时间变化，反映了市场环境的变化")
print(f"  图2：风险贡献的堆叠面积图，可以看到各因子的贡献相对稳定")
print(f"  图3：因子风险占比在 {factor_ratio.min()*100:.1f}% ~ {factor_ratio.max()*100:.1f}% 之间波动")
print(f"  图4：MCTR 随因子波动率变化，MKT 的 MCTR 通常大于 SMB")

---

## 9. 风险归因恒等式的验证

### 📐 理论

风险归因恒等式要求：

$$\sigma_p^2 = \sum_{k} \text{RC}_k + D$$

在每个月都必须成立。

In [ ]:
# ========== 验证风险归因恒等式 ==========
print("📊 验证风险归因恒等式（每个月）")
print("═" * 55)

# 计算每个月的归因总和
attribution_sum = RC_MKT_arr + RC_SMB_arr + var_specific_arr

# 与总方差的差异
diff = attribution_sum - var_total_arr

print(f"  归因总和与总方差的差异：")
print(f"    均值: {diff.mean():.10f}")
print(f"    标准差: {diff.std():.10f}")
print(f"    最大绝对值: {np.abs(diff).max():.10f}")

# 检查是否全部接近 0
all_close = np.allclose(attribution_sum, var_total_arr, atol=1e-10)
print(f"\n  所有月份归因恒等式成立: {'✅ 是' if all_close else '❌ 否'}")

print(f"\n💡 关键发现：")
print(f"  1. 风险归因恒等式在每个月都精确成立")
print(f"  2. 差异来自浮点数舍入误差，可以忽略")
print(f"  3. 这验证了我们的风险归因计算是正确的")

---

## 📌 核心概念回顾

### 📌 方差分解 (Variance Decomposition)
- **定义**: 将组合方差分解为因子风险和特异性风险
- **公式**: $\sigma_p^2 = B \Sigma_F B' + D$
- **含义**: 因子风险是系统性风险，特异性风险可以通过分散化降低
- **判断标准**: 因子风险占比越高，说明组合越受因子驱动

### 📌 边际风险贡献 (MCTR)
- **定义**: 因子暴露每增加 1 单位，组合标准差的变化量
- **公式**: $\text{MCTR}_k = \frac{(\Sigma_F \beta_p)_k}{\sigma_p}$
- **含义**: 衡量因子对风险的边际影响
- **判断标准**: MCTR 越大，说明该因子对风险的影响越大

### 📌 因子风险贡献 (RC)
- **定义**: 因子对组合方差的贡献
- **公式**: $\text{RC}_k = \beta_{p,k} \times (\Sigma_F \beta_p)_k$
- **含义**: 衡量因子对组合方差的实际贡献
- **判断标准**: RC 越大，说明该因子贡献的风险越大

### 📌 特异性风险 (Specific Risk)
- **定义**: 不能被因子解释的风险
- **公式**: $D = \sum_{i} w_i^2 \sigma_{\varepsilon,i}^2$
- **含义**: 个股的特有风险，可以通过分散化降低
- **判断标准**: 特异性风险占比越低，说明分散化效果越好

### 🔗 完整流程
```
估计因子暴露 B → 估计因子协方差 Σ_F → 计算 MCTR → 计算 RC → 验证恒等式
      ↓                ↓                ↓          ↓          ↓
   时间序列回归     样本协方差      (Σ_F×β_p)/σ_p  β×MCTR×σ_p   ΣRC+D=σ²_p
```

---

## ❌ 常见误区

### ❌ 误区 1: 风险贡献等于因子波动率
**✓ 正确理解**: 风险贡献是因子暴露和因子波动率共同决定的，$\text{RC}_k = \beta_{p,k} \times (\Sigma_F \beta_p)_k$，不是单独的因子波动率。

### ❌ 误区 2: MCTR 就是风险贡献
**✓ 正确理解**: MCTR 是边际量（每单位暴露变化的风险影响），风险贡献是总量（$\text{RC}_k = \beta_{p,k} \times \text{MCTR}_k \times \sigma_p$）。两者不同。

### ❌ 误区 3: 特异性风险不能被降低
**✓ 正确理解**: 特异性风险可以通过分散化（增加持股数量）降低。当 $N \to \infty$ 时，$\sum w_i^2 \sigma_{\varepsilon,i}^2 \to 0$。

### ❌ 误区 4: 因子风险占比是固定的
**✓ 正确理解**: 因子风险占比会随市场环境变化。当因子波动率增大时，因子风险占比会上升。

### ❌ 误区 5: 风险归因和收益归因的因子贡献相同
**✓ 正确理解**: 风险归因和收益归因使用不同的公式，得到的因子贡献不同。风险归因关注方差的来源，收益归因关注收益的来源。